# Timestamp extraction helper
This notebook inspects EXIF and filesystem timestamps for sample frames from the experiment configured in `configs/opencv_tracker_v3.yaml`.


In [ ]:
from datetime import datetime
from pathlib import Path
from typing import Optional

import yaml
from PIL import Image, ExifTags

CONFIG_PATH = Path('../configs/opencv_tracker_v3.yaml').resolve()
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')) or {}
input_cfg = config.get('input', {})
config_dir = CONFIG_PATH.parent

def resolve_path(path_like, base_dir: Path) -> Path:
    candidate = Path(path_like)
    return candidate if candidate.is_absolute() else (base_dir / candidate).resolve()

def get_base_dir() -> Path:
    base_dir = input_cfg.get('base_dir')
    if not base_dir:
        raise ValueError('base_dir is not set in the config')
    return resolve_path(base_dir, config_dir)

def get_example_frame() -> Path:
    base_dir = get_base_dir()
    candidates = sorted(base_dir.glob('**/*.bmp'))
    if not candidates:
        raise FileNotFoundError('No BMP frames found under {}'.format(base_dir))
    return candidates[0]

base_dir = get_base_dir()
example_frame = get_example_frame()

def get_exif_timestamp(path: Path) -> Optional[datetime]:
    try:
        img = Image.open(path)
        info = img._getexif() or {}
    except Exception:
        return None
    for tag, value in info.items():
        decoded = ExifTags.TAGS.get(tag, tag)
        if decoded in ('DateTimeOriginal', 'DateTime'):
            try:
                return datetime.strptime(value, '%Y:%m:%d %H:%M:%S')
            except ValueError:
                continue
    return None

def get_filesystem_timestamp_seconds(path: Path) -> float:
    return path.stat().st_mtime

print('Using base directory:', base_dir)
print('Sample frame for timestamp checks:', example_frame)
print('Frame belongs to config:', CONFIG_PATH.name)


In [ ]:
print('EXIF ts:', get_exif_timestamp(example_frame))
print('filesystem ts (datetime):', datetime.fromtimestamp(get_filesystem_timestamp_seconds(example_frame)))
print('filesystem ts (seconds):', get_filesystem_timestamp_seconds(example_frame))
